# OncoVisionAI - Grad-CAM Visualization

This notebook demonstrates the **Explainable AI** component using Grad-CAM (Gradient-weighted Class Activation Mapping).

## What is Grad-CAM?

Grad-CAM generates visual explanations by highlighting the regions of an image that were most important for the model's prediction. This builds trust and allows health workers to verify the AI's reasoning.

## Contents
1. Load trained model
2. Load test images
3. Generate Grad-CAM heatmaps
4. Visualize explanations
5. Batch processing

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from tensorflow import keras
from pathlib import Path

from src.gradcam import GradCAM, batch_generate_gradcam
from src.data_preprocessing import CancerDataPreprocessor

# Set style
plt.rcParams['figure.figsize'] = (16, 10)

print("✓ Imports successful")

## 1. Load Trained Model

In [ ]:
# Load model
model_path = '../models/saved_models/oncovision_multimodal.h5'
model = keras.models.load_model(model_path)

print(f"✓ Model loaded from {model_path}")
print(f"  Total parameters: {model.count_params():,}")

## 2. Load Test Data

In [ ]:
# Load clinical data
clinical_df = pd.read_csv('../data/clinical_data.csv')

# Sample some test cases
num_samples = 6
sample_df = clinical_df.sample(num_samples, random_state=42)

print(f"Selected {num_samples} samples for Grad-CAM visualization")
sample_df[['image_id', 'age', 'lesion_size_mm', 'label']]

## 3. Initialize Grad-CAM

In [ ]:
# Initialize Grad-CAM
gradcam = GradCAM(model)

print(f"✓ Grad-CAM initialized")
print(f"  Target layer: {gradcam.layer_name}")

## 4. Generate Grad-CAM for Single Image

In [ ]:
# Select first sample
sample = sample_df.iloc[0]

# Load and preprocess image
img_path = f"../data/raw/images/{sample['image_id']}"
preprocessor = CancerDataPreprocessor(img_size=(224, 224))

# Load original image
original_img = cv2.imread(img_path)
original_img = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)

# Preprocess for model
processed_img = preprocessor.load_and_preprocess_image(img_path, augment=False)

# Prepare clinical data
clinical_data = np.array([[
    sample['age'],
    sample['symptom_duration_months'],
    sample['family_history'],
    sample['pain_score'],
    sample['lesion_size_mm']
]])

# Normalize clinical data (approximate)
clinical_mean = np.array([50.0, 10.0, 0.5, 5.0, 15.0])
clinical_std = np.array([18.0, 10.0, 0.5, 3.0, 10.0])
clinical_normalized = (clinical_data - clinical_mean) / clinical_std

print(f"Processing image: {sample['image_id']}")
print(f"True label: {'Malignant' if sample['label'] == 1 else 'Benign'}")

In [ ]:
# Generate Grad-CAM explanation
result = gradcam.generate_explanation(
    processed_img,
    clinical_normalized[0],
    original_img,
    class_names=['Benign', 'Malignant'],
    save_path='../outputs/gradcam_single_example.png'
)

print(f"\n✓ Grad-CAM generated!")
print(f"  Predicted: {result['predicted_label']}")
print(f"  Confidence: {result['confidence']*100:.1f}%")
print(f"  Benign probability: {result['predictions'][0]*100:.1f}%")
print(f"  Malignant probability: {result['predictions'][1]*100:.1f}%")

## 5. Visualize Multiple Samples

In [ ]:
# Process all samples
fig, axes = plt.subplots(num_samples, 3, figsize=(18, num_samples * 5))

for idx, (_, sample) in enumerate(sample_df.iterrows()):
    # Load image
    img_path = f"../data/raw/images/{sample['image_id']}"
    original_img = cv2.imread(img_path)
    original_img = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)
    processed_img = preprocessor.load_and_preprocess_image(img_path, augment=False)
    
    # Prepare clinical data
    clinical_data = np.array([[
        sample['age'],
        sample['symptom_duration_months'],
        sample['family_history'],
        sample['pain_score'],
        sample['lesion_size_mm']
    ]])
    clinical_normalized = (clinical_data - clinical_mean) / clinical_std
    
    # Generate Grad-CAM
    heatmap, _ = gradcam.compute_heatmap(processed_img, clinical_normalized[0])
    overlay = gradcam.overlay_heatmap(heatmap, original_img)
    
    # Get prediction
    pred = model.predict([np.expand_dims(processed_img, 0), clinical_normalized], verbose=0)[0]
    pred_class = np.argmax(pred)
    pred_label = 'Malignant' if pred_class == 1 else 'Benign'
    true_label = 'Malignant' if sample['label'] == 1 else 'Benign'
    
    # Plot
    axes[idx, 0].imshow(original_img)
    axes[idx, 0].set_title(f'Original\nTrue: {true_label}', fontsize=12, fontweight='bold')
    axes[idx, 0].axis('off')
    
    axes[idx, 1].imshow(heatmap, cmap='jet')
    axes[idx, 1].set_title('Grad-CAM Heatmap', fontsize=12, fontweight='bold')
    axes[idx, 1].axis('off')
    
    color = 'green' if pred_class == sample['label'] else 'red'
    axes[idx, 2].imshow(overlay)
    axes[idx, 2].set_title(
        f'Prediction: {pred_label}\nConfidence: {pred[pred_class]*100:.1f}%',
        fontsize=12,
        fontweight='bold',
        color=color
    )
    axes[idx, 2].axis('off')

plt.tight_layout()
plt.savefig('../outputs/gradcam_multiple_samples.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Multiple Grad-CAM visualizations generated")

## 6. Analyze Grad-CAM Patterns

In [ ]:
# Compare benign vs malignant attention patterns
benign_samples = sample_df[sample_df['label'] == 0].head(3)
malignant_samples = sample_df[sample_df['label'] == 1].head(3)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Benign samples
for idx, (_, sample) in enumerate(benign_samples.iterrows()):
    img_path = f"../data/raw/images/{sample['image_id']}"
    original_img = cv2.imread(img_path)
    original_img = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)
    processed_img = preprocessor.load_and_preprocess_image(img_path, augment=False)
    
    clinical_data = np.array([[
        sample['age'], sample['symptom_duration_months'],
        sample['family_history'], sample['pain_score'], sample['lesion_size_mm']
    ]])
    clinical_normalized = (clinical_data - clinical_mean) / clinical_std
    
    heatmap, _ = gradcam.compute_heatmap(processed_img, clinical_normalized[0])
    overlay = gradcam.overlay_heatmap(heatmap, original_img)
    
    axes[0, idx].imshow(overlay)
    axes[0, idx].set_title(f'Benign Sample {idx+1}', fontsize=12, fontweight='bold', color='green')
    axes[0, idx].axis('off')

# Malignant samples
for idx, (_, sample) in enumerate(malignant_samples.iterrows()):
    img_path = f"../data/raw/images/{sample['image_id']}"
    original_img = cv2.imread(img_path)
    original_img = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)
    processed_img = preprocessor.load_and_preprocess_image(img_path, augment=False)
    
    clinical_data = np.array([[
        sample['age'], sample['symptom_duration_months'],
        sample['family_history'], sample['pain_score'], sample['lesion_size_mm']
    ]])
    clinical_normalized = (clinical_data - clinical_mean) / clinical_std
    
    heatmap, _ = gradcam.compute_heatmap(processed_img, clinical_normalized[0])
    overlay = gradcam.overlay_heatmap(heatmap, original_img)
    
    axes[1, idx].imshow(overlay)
    axes[1, idx].set_title(f'Malignant Sample {idx+1}', fontsize=12, fontweight='bold', color='red')
    axes[1, idx].axis('off')

plt.suptitle('Grad-CAM Attention Patterns: Benign vs Malignant', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/gradcam_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Comparison visualization generated")

## 7. Batch Generation for All Test Samples

In [ ]:
# This will generate Grad-CAM for multiple test samples
# Uncomment to run (may take a few minutes)

# from src.data_preprocessing import CancerDataPreprocessor
# 
# preprocessor = CancerDataPreprocessor()
# dataset = preprocessor.prepare_multimodal_dataset(
#     '../data/raw/images',
#     '../data/clinical_data.csv'
# )
# 
# # Load test images
# test_images = []
# test_originals = []
# for img_path in dataset['test']['image_paths'][:20]:
#     img = preprocessor.load_and_preprocess_image(img_path, augment=False)
#     orig = cv2.imread(img_path)
#     orig = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)
#     test_images.append(img)
#     test_originals.append(orig)
# 
# test_images = np.array(test_images)
# test_clinical = dataset['test']['clinical'][:20]
# 
# # Generate batch Grad-CAM
# batch_generate_gradcam(
#     model,
#     test_images,
#     test_clinical,
#     test_originals,
#     output_dir='../outputs/gradcam_batch',
#     num_samples=20
# )

print("Batch generation code ready (uncomment to run)")

## Key Insights from Grad-CAM

### What Grad-CAM Shows:

1. **Red/Yellow Regions**: Areas that most influenced the "Malignant" prediction
2. **Blue/Purple Regions**: Areas with less influence
3. **Focused Attention**: Model looks at specific lesion features (borders, texture, color)

### Benefits for Rural Healthcare:

- ✅ **Trust Building**: Health workers can verify AI reasoning
- ✅ **Education**: Shows what features indicate malignancy
- ✅ **Quality Control**: Identifies when model focuses on wrong areas
- ✅ **Transparency**: No "black box" - clear visual explanation

## Next Steps

1. Use these visualizations in your hackathon demo
2. Show judges how Grad-CAM builds trust
3. Explain how this helps rural health workers
4. Integrate into mobile app for real-time explanations